# DEMO: Unity Catalog 3 Level Namespace, Data Discovery and Data Lineage

In this demo you will explore how Databricks organises and governs data using **Unity Catalog**. You will:

1. See how every data object lives within a **three-level namespace** (`catalog.schema.object`)
2. Discover data using both **SQL commands** and the **Catalog Explorer UI**
3. Explore **automatic data lineage** — see how data flows between tables, views and notebooks

---

## Step 1: Run Setup

The cell below runs a setup notebook that creates demo tables and views in your workspace. These objects will be used throughout the rest of this demo.

Click the **Run** button (▶) on the cell below to execute it.

In [0]:
%run ./Setup

---

## Step 2: Data Discovery with Catalog Explorer (UI)

SQL commands are powerful for programmatic discovery, but Databricks also provides a visual interface called **Catalog Explorer**. Follow these steps to explore your data in the UI.

### 2a. Open Catalog Explorer

1. Look at the **left sidebar** of the Databricks workspace
2. Click the **Catalog** icon (it looks like a database/cylinder icon with a magnifying glass) — it is labelled "Catalog" when you hover over it
3. The Catalog Explorer panel opens, showing a tree of catalogs you have access to

### 2b. Browse the Three-Level Namespace in the UI

1. In the Catalog Explorer panel, you will see a list of catalogs. Find and click your **workspace catalog** (it will match the catalog name printed by the setup cell above)
2. The catalog expands to show its **schemas**. Find and click the schema that starts with `demo_lineage_` followed by your username
3. The schema expands to show **Tables** and **Views**. Click on **Tables** to see `customers` and `orders`
4. Click the **`customers`** table name

### 2c. Explore Table Details in the UI

Once you've clicked on the `customers` table, you'll see a detailed page with several tabs:

1. **Overview** tab (default)
   - Shows the table comment at the top
   - Displays all columns with their types and comments
   - Shows tags if any have been applied

2. **Sample Data** tab
   - Click this tab to preview actual rows from the table without writing any SQL
   - This is equivalent to running `SELECT * FROM table LIMIT ...`

3. **Details** tab
   - Shows metadata like the owner, creation date, storage location, and table type
   - Similar to what `DESCRIBE EXTENDED` showed you in SQL

4. **Permissions** tab
   - Shows who has access to this table and what privileges they have
   - This is the governance layer in action

### 2d. Use the Search Bar for Discovery

1. Click the **Search bar** at the very top of the Databricks workspace (or press `Ctrl+P` / `Cmd+P`)
2. Type `customers` and press Enter
3. Notice the search finds your table along with any other matching objects
4. The search works across table names, column names, and comments — so searching for "account balance" would also find the `customers` table because of its column comment

## Step 3: Data Lineage

Unity Catalog automatically captures **data lineage** — it tracks how data flows from source tables through transformations into downstream tables and views. This happens automatically when you run queries on Databricks.

The setup notebook already generated lineage by:
- Creating `customers` from `samples.tpch.customer` + `samples.tpch.nation`
- Creating `orders` from `samples.tpch.orders`  
- Creating `revenue_by_nation` from `customers` + `orders`

Let's view this lineage in Catalog Explorer.

### 3a. View Table Lineage

1. Open **Catalog Explorer** (click the Catalog icon in the left sidebar)
2. Navigate to your demo schema: **your workspace catalog → demo_lineage_\<your username\> → Tables**
3. Click on the **`revenue_by_nation`** view
4. Click the **Lineage** tab (next to Overview, Sample Data, Details, etc.)
5. You will see a panel showing **related tables** — these are the upstream sources and downstream consumers

### 3b. View the Lineage Graph

1. While on the Lineage tab, click the **"See Lineage Graph"** button
2. An interactive graph appears showing how data flows:
   - **Upstream** (left side): `samples.tpch.customer`, `samples.tpch.nation`, and `samples.tpch.orders` are the original source tables
   - **Intermediate** (middle): your `customers` and `orders` tables were derived from the sources
   - **Downstream** (right side): `revenue_by_nation` consumes both `customers` and `orders`
3. Click the **expand icon** (▶) on any node to reveal additional connections if they exist
4. Click on a **connecting edge** (the line between nodes) to open the Lineage details panel, which shows source and target column mappings

### 3c. View Column-Level Lineage

1. While in the lineage graph, click on a specific **column name** within a node (for example, click `total_revenue` in the `revenue_by_nation` node)
2. The graph highlights the upstream columns that feed into it — in this case, you will see it traces back to `total_price` in the `orders` table
3. This is **column-level lineage** — it tells you exactly where each column’s data originates

### 3d. View Notebook Lineage

1. On the Lineage tab (not the graph), look for the **Notebooks** section or filter
2. You should see the `demo_1_setup` notebook listed as a producer of this table
3. This tells you which notebook or job created or modified the data

> **Key difference from Power BI:** Power BI’s Lineage View only shows connections between Power BI artifacts (dataflows → datasets → reports). Unity Catalog lineage captures the complete data flow including source tables, notebooks, jobs, pipelines, and dashboards — all the way down to individual columns. It is captured automatically from any query run on Databricks.

## Step 4: Programmatic Data Discovery and Access

Every data object in Unity Catalog lives within a **three-level hierarchy**:

```
catalog.schema.object
```

For example: `samples.tpch.orders`

| Level | Purpose |
|---|---|
| **Catalog** | Top-level container — often maps to an environment (dev/prod) or business unit |
| **Schema** | A logical grouping of related tables, views, and other objects |
| **Object** | A table, view, volume, or function you can query |

Let's explore each level with SQL commands.

In [0]:
%sql
-- List all catalogs you have access to in this workspace
SHOW CATALOGS;

You should see at least two catalogs:
- Your **workspace catalog** (the default catalog for this workspace)
- **`samples`** — a read-only catalog provided by Databricks with sample datasets

Now let's look inside the `samples` catalog to see its schemas.

In [0]:
%sql
-- List all schemas in the samples catalog
SHOW SCHEMAS IN samples;

In [0]:
%sql
-- List all tables in the samples.tpch schema
SHOW TABLES IN samples.tpch;

Notice the pattern: **catalog → schema → tables**. This is the three-level namespace in action. Every table you query uses this full path.

Now let's look at the schema the setup notebook created for you:

In [0]:
%sql
-- Show schemas in your workspace catalog (the current catalog)
-- Since the setup notebook set your catalog context, this shows schemas in that catalog
SHOW SCHEMAS;

In [0]:
%sql
-- List the tables and views in your demo schema
SHOW TABLES;

You should see three objects: the `customers` table, the `orders` table, and the `revenue_by_nation` view — all created by the setup notebook.

---

## Step 3: Data Discovery with SQL

Now that you can see what objects exist, let's dig deeper into individual tables. These commands let you understand what data a table contains without having to query it.

In [0]:
%sql
-- Describe the structure of the customers table (columns, types, and comments)
DESCRIBE TABLE samples.tpch.nation;

Notice each column shows its **name**, **data type**, and **comment**. Column comments help other users understand what the data means without needing to ask the person who created the table.

For even more detail about a table, use `DESCRIBE EXTENDED`:

In [0]:
%sql
-- Get extended details: owner, location, table properties, and more
DESCRIBE TABLE EXTENDED samples.tpch.nation;

Scroll through the results. You can see:
- The **Owner** of the table (you!)
- The **table comment** describing what it contains
- The **storage location** where the Delta files live
- The **table type** (MANAGED means Unity Catalog manages the storage for you)

You can also query any table using its fully qualified three-level name:

In [0]:
%sql
-- Query data using the fully qualified name: catalog.schema.table
SELECT * FROM samples.tpch.nation;

Before you run the next cell you'll notice this query references `revenue_by_nation` with just a single name — no catalog or schema prefix needed. This works because the setup notebook ran two commands at the start:


```sql
USE CATALOG <your_workspace_catalog>;
USE SCHEMA demo_lineage_<your_username>;
```

These commands set a **default catalog and schema** for your session. From this point on, any unqualified object name (like `revenue_by_nation`) is automatically resolved against that default context — equivalent to writing the full `catalog.schema.revenue_by_nation` path.

In [0]:
%sql
-- Query data from your demo schema (current schema context means you don't need the full path)
SELECT nation, market_segment, total_orders, total_revenue 
FROM revenue_by_nation 
ORDER BY total_revenue DESC 
LIMIT 10;

In [0]:
%sql
-- You can also explore lineage programmatically by querying the data directly
-- This query shows the view definition, which reveals its upstream dependencies
SHOW CREATE TABLE revenue_by_nation;

The `SHOW CREATE TABLE` output shows you the SQL definition of the view, which reveals its dependencies on the `customers` and `orders` tables.

---

## Summary

In this demo you have:

1. **Discovered data with the Catalog Explorer UI** — browsing the hierarchy, viewing table details, previewing sample data, and using global search
1. **Viewed automatic data lineage** — seeing how data flows between tables and views, down to the column level, captured automatically by Unity Catalog
1. **Explored the three-level namespace** — every object in Databricks is addressed as `catalog.schema.object`
1. **Discovered data with SQL** — using `SHOW CATALOGS`, `SHOW SCHEMAS`, `SHOW TABLES`, `DESCRIBE TABLE`, and `DESCRIBE EXTENDED`


In [0]:
%sql
-- Cleanup: uncomment and run the line below ONLY if you want to remove the demo objects
-- DROP SCHEMA IF EXISTS IDENTIFIER(CONCAT(current_catalog(), '.', current_schema())) CASCADE;